<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 5 章：练习题解答

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.22.2


&nbsp;
## 练习 5.1：在自一致性中使用启发式评分器作为平局决胜方法

- 有很多方式可以实现这一点
- 最简单的方式可能是在自一致性函数外部处理，使用返回的字典（例如，类似于我们在练习 4.4 中实现平局处理时所做的，我们直接将其添加到 `evaluate_math500_stream` 函数中）
- 相关行如下所示

```python
# ...
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with heuristic_score
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            # ...

    # ...
    return num_correct, num_examples, acc
```

- 相比第 3 章的基线和第 4 章的自一致性，改进如下所示

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 第 4 章基线 + CoT 提示    | 基座  | 33.4%    | 129.2 min |
| 2 | 自一致性 (n=3) + 多数投票   | 基座  | 43.2%    | 328.2 min |
| 3 | 自一致性 (n=3) + 启发式评分       | 基座  | 43.4%    | 326.5 min |
| 4 | 自一致性 (n=3) + 平均 logprob    | 基座  | 44.8%    | 327.7 min |

- 表中显示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上使用 "cuda" GPU（DGX Spark）计算的

- 为了方便，你可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 中的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_consistency_scorer_math500.py) 脚本

- 但请注意，如 [#159](https://github.com/rasbt/reasoning-from-scratch/issues/159) 中所讨论的，我们基于启发式评分决定多数获胜者，但只考虑每个多数对中的第一个实例
- 例如

&nbsp;
## 练习 5.2：在 Best-of-N 设置中使用启发式评分器

- Best-of-N 类似于自一致性，都是生成多个答案
- 但是，我们不是基于多数投票选择最终答案，而是使用评分函数（如 `heuristic_score`）对所有答案进行评分并返回得分最高的答案
- 有多种方式可以实现此行为，但最简单的方式可能是使用第 4 章中现有的自一致性函数作为模板，并替换为 `heuristic_score`，如下所示

```python
# ...

from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- 结果如下所示

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 基线 + 思维链提示 | 基座  | 33.4%    | 129.2 min |
| 2 | Best-of-N (n=3) + 启发式评分              | 基座  | 40.6%    | 327.7 min |
| 3 | Best-of-N (n=3) + 平均 logprob           | 基座  | 43.2%    | 330.2 min |

- 表中显示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上使用 "cuda" GPU（DGX Spark）计算的

- 为了方便，你可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 中的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.3：在自一致性中使用 logprob 评分器作为平局决胜方法

- 代码类似于练习 5.1，只是将 `heuristic_score` 替换为 `avg_logprob_answer`

```python
# ...
# from reasoning_from_scratch.ch05 import heuristic_score
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with avg_logprob_answer
            else:
                best = None
                best_score = float("-inf")
            
                # Consider all members of each majority group
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            # ...

    # ...
    return num_correct, num_examples, acc
```

- 相比第 3 章的基线和第 4 章的自一致性，改进如下所示

|   | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 基线 + 思维链提示 | 基座  | 33.4%    | 129.2 min |
| 2 | 自一致性 (n=3) + 多数投票   | 基座  | 43.2%    | 328.2 min |
| 3 | 自一致性 (n=3) + 启发式评分       | 基座  | 43.4%    | 326.5 min |
| 4 | 自一致性 (n=3) + 平均 logprob     | 基座  | 44.8%    | 327.7 min |

- 表中显示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上使用 "cuda" GPU（DGX Spark）计算的

- 为了方便，你可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 中的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.4：在 Best-of-N 设置中使用 logprob 评分器

- 要使用 logprob 评分器实现 Best-of-N，我们可以使用练习 5.2 的代码，将 `heuristic_score` 替换为 `avg_logprob_answer`：

```python

from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- 结果如下所示

| # | 方法                                   | 模型 | 准确率 | 时间      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 基线 + 思维链提示 | 基座  | 33.4%    | 129.2 min |
| 2 | Best-of-N (n=3) + 启发式评分              | 基座  | TBD      | TBD       |
| 3 | Best-of-N (n=3) + 平均 logprob           | 基座  | TBD      | TBD       |

- 表中显示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上使用 "cuda" GPU（DGX Spark）计算的

- 为了方便，你可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 中的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.5：使用启发式评分进行自我优化

- 使用 `heuristic_score` 实际上比使用 logprob 评分更简单，我们只需要更改以下代码：

```python
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)


torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 更新后的代码为：

```python
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # NEW
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 使用启发式评分器的结果显示在第 4、5 和 10 行：

|    | 方法                 | 评分方式       | 迭代次数 | 模型      | 准确率 | 时间      |
|----|------------------------|---------------|------------|------------|----------|-----------|
| 1  | 基线（第 3 章）   | -             | -          | 基座       | 15.2%    | 10.1 min  |
| 2  | 自我优化        | 无          | 1          | 基座       | 25.0%    | 84.8 min  |
| 3  | 自我优化        | 无          | 2          | 基座       | 22.0%    | 165.4 min |
| 4  | 自我优化        | 启发式     | 1          | 基座       | 21.6%    | 84.7 min  |
| 5  | 自我优化        | 启发式     | 2          | 基座       | 20.8%    | 151.4 min |
| 6  | 自我优化        | 平均 logprob  | 1          | 基座       | 21.4%    | 85.3 min  |
| 7  | 自我优化        | 平均 logprob  | 2          | 基座       | 22.0%    | 165.3 min |
|    |                        |               |            |            |          |           |
| 8  | 基线（第 3 章）   | -             | -          | 推理  | 48.2%    | 182.1 min |
| 9  | 自我优化        | 无          | 1          | 推理  | 56.6%    | 498.8 min |
| 10 | 自我优化        | 启发式     | 1          | 推理  | 57.8%    | 498.6 min |
| 11 | 自我优化        | 平均 logprob  | 1          | 推理  | 48.4%    | 499.7 min |

- 表中显示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上使用 "cuda" GPU（DGX Spark）计算的
- 为了方便，你可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 中的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_refinement_math500.py) 脚本